# Session 5 — Project Tasks — P4A 2026
**Group 23**

This notebook implements the local LLM and data agent tasks for Session 5 of the group project.

## Setup
Initialize ChatOllama with the local `llama3.2` model and import required libraries.

In [ ]:
import os
import json
import pandas as pd
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langchain.agents import create_agent

# Configure local LLM model
MODEL_NAME = "llama3.2"
print(f"Connecting to local Ollama with model: {MODEL_NAME}")

# Initialize ChatOllama
llm = ChatOllama(model=MODEL_NAME, temperature=0)


## Task 1 — Interpret findings with an LLM

Pass findings from Session 4 (Top Industries and Medical Insurance offerings) to the local LLM and interpret them.

In [ ]:
# Prepare findings from Session 4
findings = """
Finding 1: Top industries by full-time posting count:
1. Hospitals and Health Care: 4,557 postings
2. Financial Services: 2,394 postings
3. Retail: 2,383 postings
4. Staffing and Recruiting: 2,334 postings

Finding 2: Medical Insurance benefits by industry (Top industries offering Medical Insurance):
1. Hospitals and Health Care: 235 postings
2. Staffing and Recruiting: 130 postings
3. Financial Services: 121 postings
4. Construction: 116 postings

Note: Retail is completely absent from the top 10 list for medical insurance offerings, despite being the 3rd largest industry by overall postings.
"""

system_prompt = SystemMessage(
    content="You are a labor economist and business strategist. Your goal is to analyze job posting data and extract meaningful, non-trivial insights."
)

user_prompt = HumanMessage(
    content=f"Analyze the following findings from our LinkedIn Job Postings dataset. Specifically, interpret why the Retail industry has such a high listing volume but is completely absent from the top 10 list of industries offering Medical Insurance benefits:\n\n{findings}"
)

messages = [system_prompt, user_prompt]
response = llm.invoke(messages)

print("--- Data Passed to Model ---")
print(findings)
print("\n--- Prompts Used ---")
print(f"System Prompt: {system_prompt.content}")
print(f"User Prompt: {user_prompt.content}")
print("\n--- Model's Response ---")
print(response.content)


**Value Assessment:**
Does the response add value beyond what the data already shows, or is it just restating the numbers?

*Replace this text with your assessment.*

## Task 2 — Structured output

Ask the LLM to summarize the retail vs. healthcare benefits discrepancy into a structured JSON object.

In [ ]:
structured_prompt = f"""
Summarize the discrepancy between high posting counts and low medical insurance offerings in the Retail industry from our dataset.
You must return the output as a valid JSON object matching the following schema. Do not return any other text or markdown block, only the JSON itself.

{{
  "highest_posting_industry": "string",
  "discrepancy_industry": "string",
  "observed_gap_description": "string",
  "proposed_reason": "string",
  "actionability_level": "string (High/Medium/Low)"
}}

Findings:
{findings}
"""

response_structured = llm.invoke([HumanMessage(content=structured_prompt)])
raw_response = response_structured.content.strip()
print("--- Raw Model Response ---")
print(raw_response)

# Strip markdown backticks if present
if raw_response.startswith("```json"):
    raw_response = raw_response[7:]
elif raw_response.startswith("```"):
    raw_response = raw_response[3:]
if raw_response.endswith("```"):
    raw_response = raw_response[:-3]
raw_response = raw_response.strip()

print("\n--- Parsed Output ---")
try:
    parsed_json = json.loads(raw_response)
    print(json.dumps(parsed_json, indent=2))
except Exception as e:
    print(f"Failed to parse JSON: {e}")


**JSON Quality Assessment:**
Did the model return valid JSON on the first try? If not, what did you do?

*Replace this text with your assessment.*

## Task 3 — Multi-turn conversation

Hold a 3-turn conversation about findings, building context as the turns progress.

In [ ]:
conversation_history = [
    SystemMessage(content="You are a helpful data science assistant guiding a business analyst through LinkedIn job postings research.")
]

turns = [
    "Based on our job postings dataset, what are the top industries by volume of listings, and which of these are the most generous in providing medical insurance?",
    "Interesting. Why do you think Retail has so many job postings but almost no postings listing medical insurance compared to Hospitals and Health Care or Financial Services?",
    "Given this benefit gap in retail, how can we use this data to advise a job board or recruitment agency on how to target employer outreach or improve listing conversions?"
]

for i, turn in enumerate(turns, 1):
    print(f"\n=== Turn {i} ===")
    print(f"User: {turn}")
    conversation_history.append(HumanMessage(content=turn))
    
    response = llm.invoke(conversation_history)
    print(f"AI: {response.content}")
    conversation_history.append(AIMessage(content=response.content))

# Print message history summary
print("\n=== Message History Summary ===")
for msg in conversation_history:
    role = "System" if isinstance(msg, SystemMessage) else ("User" if isinstance(msg, HumanMessage) else "AI")
    preview = msg.content[:80].replace("\n", " ") + "..."
    print(f"- {role}: {preview}")


**Context Assessment:**
Did the model maintain context correctly across the three turns?

*Replace this text with your assessment.*

## Task 4 — Build a data agent

Define custom tools that query the output files directly and run a LangChain Agent Graph to demonstrate reasoning.

In [ ]:
@tool
def get_industry_posting_count(industry_name: str) -> str:
    """Queries the dataset and returns the number of full-time postings for the given industry name.
    Useful for answering questions about industry posting volume.
    """
    try:
        df = pd.read_csv("output/industry_posting_counts.csv")
        match = df[df['industry_name'].str.lower() == industry_name.lower()]
        if not match.empty:
            count = match.iloc[0]['posting_count']
            return f"Industry '{industry_name}' has {count} full-time job postings."
        else:
            return f"Industry '{industry_name}' was not found in the dataset."
    except Exception as e:
        return f"Error querying dataset: {str(e)}"

@tool
def get_top_industries(n: int = 5) -> str:
    """Returns the top N industries by number of full-time postings.
    Useful for answering questions about which industries are the largest by volume.
    """
    try:
        df = pd.read_csv("output/industry_posting_counts.csv")
        top_n = df.head(n)
        result = "Top industries by posting count:\n"
        for i, row in top_n.iterrows():
            result += f"{i+1}. {row['industry_name']}: {row['posting_count']} postings\n"
        return result
    except Exception as e:
        return f"Error loading top industries: {str(e)}"

@tool
def get_medical_insurance_benefits() -> str:
    """Returns the top industries offering medical insurance benefits and their posting counts.
    Useful for answering questions about which industries are most generous with medical insurance.
    """
    return (
        "Top industries by medical insurance benefits postings:\n"
        "1. Hospitals and Health Care: 235 postings\n"
        "2. Staffing and Recruiting: 130 postings\n"
        "3. Financial Services: 121 postings\n"
        "4. Construction: 116 postings\n"
        "5. IT Services and IT Consulting: 92 postings\n"
        "6. Manufacturing: 91 postings\n"
        "7. Insurance: 77 postings\n"
        "8. Defense and Space Manufacturing: 75 postings\n"
        "9. Business Consulting and Services: 71 postings\n"
        "10. Law Practice: 71 postings"
    )

@tool
def get_top_countries(n: int = 5) -> str:
    """Returns the top N countries by number of companies in the dataset.
    Useful for answering questions about the geographic distribution of companies.
    """
    try:
        with open("output/top_countries.json", "r") as f:
            data = json.load(f)
        top_countries = data.get("top_countries", [])[:n]
        result = "Top countries by number of companies:\n"
        for i, country in enumerate(top_countries):
            result += f"{i+1}. {country['country']}: {country['count']} companies\n"
        return result
    except Exception as e:
        return f"Error loading top countries: {str(e)}"

# List of tools
tools = [get_industry_posting_count, get_top_industries, get_medical_insurance_benefits, get_top_countries]

# Initialize Agent Graph using create_agent from langchain.agents
agent_executor = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful data assistant. Use your tools to answer questions about the LinkedIn job postings and companies dataset."
)

# Test agent trace
question = "How many full-time postings does the Financial Services industry have, and which are the top 3 countries by company count in our dataset?"
print(f"Question: {question}\n")

inputs = {"messages": [{"role": "user", "content": question}]}
response = agent_executor.invoke(inputs)

# Print the trace
print("--- Agent Execution Trace ---")
for msg in response["messages"]:
    role = msg.__class__.__name__.replace("Message", "")
    print(f"\n[{role}]:")
    print(msg.content)
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"Tool Calls: {json.dumps(msg.tool_calls, indent=2)}")

print("\nFinal Answer:\n", response["messages"][-1].content)


Running on agent.py

conda activate p4a2026
python project_group23/agent.py

Connecting to local Ollama with model: llama3.2

--- Data Agent CLI ---
Ask the agent questions about your LinkedIn Job Postings findings.
Press Ctrl+C or type 'exit' to quit.

Demo question: How many full-time postings does the Financial Services industry have, and which are the top 3 countries by company count in our dataset?
Running demo...

--- Agent Execution Trace ---

[Human]:
How many full-time postings does the Financial Services industry have, and which are the top 3 countries by company count in our dataset?

[AI]:

Tool Calls: [
  {
    "name": "get_industry_posting_count",
    "args": {
      "industry_name": "Financial Services"
    },
    "id": "c0d8590c-1c5e-4c6a-ab3f-f363329c1804",
    "type": "tool_call"
  },
  {
    "name": "get_top_countries",
    "args": {
      "n": "3"
    },
    "id": "984cde2e-9cc2-402a-b95a-b8a8e864ae9d",
    "type": "tool_call"
  }
]

[Tool]:
Industry 'Financial Services' has 2394 full-time job postings.

[Tool]:
Top countries by number of companies:
1. US: 21635 companies
2. 0: 727 companies
3. GB: 620 companies


[AI]:
The Financial Services industry has 2394 full-time job postings.

The top 3 countries by company count in the Financial Services industry are:

1. United States (US) with 21,635 companies
2. United Kingdom (GB) with 620 companies
3. India (0) with 727 companies

Final Answer:
 The Financial Services industry has 2394 full-time job postings.

The top 3 countries by company count in the Financial Services industry are:

1. United States (US) with 21,635 companies
2. United Kingdom (GB) with 620 companies
3. India (0) with 727 companies

Question: ^C
Exiting. Goodbye!
(p4a2026) gabrielpiazzalunga@Mac p4a % conda activate p4a2026
python project_group23/agent.py

Connecting to local Ollama with model: llama3.2

--- Data Agent CLI ---
Running your 3 project questions to generate execution traces...


=======================================================
PROJECT QUESTION 1: Based on our job postings dataset, what are the top industries by volume of listings, and which of these are the most generous in providing medical insurance?
=======================================================
Running agent...

--- Agent Execution Trace ---

[Human]:
Based on our job postings dataset, what are the top industries by volume of listings, and which of these are the most generous in providing medical insurance?

[AI]:

Tool Calls: [
  {
    "name": "get_top_industries",
    "args": {
      "industry_name": [],
      "n": "10"
    },
    "id": "cf2d3585-44f5-46a3-986b-d761681c3f3f",
    "type": "tool_call"
  },
  {
    "name": "get_medical_insurance_benefits",
    "args": {},
    "id": "616f87ee-6526-4961-9bd8-68f47fc1911e",
    "type": "tool_call"
  }
]

[Tool]:
Top industries by posting count:
1. Hospitals and Health Care: 4557 postings
2. Financial Services: 2394 postings
3. Retail: 2383 postings
4. Staffing and Recruiting: 2334 postings
5. IT Services and IT Consulting: 2063 postings
6. Software Development: 1113 postings
7. Construction: 940 postings
8. Manufacturing: 865 postings
9. Banking: 700 postings
10. Telecommunications: 656 postings


[Tool]:
Top industries by medical insurance benefits postings:
1. Hospitals and Health Care: 235 postings
2. Staffing and Recruiting: 130 postings
3. Financial Services: 121 postings
4. Construction: 116 postings
5. IT Services and IT Consulting: 92 postings
6. Manufacturing: 91 postings
7. Insurance: 77 postings
8. Defense and Space Manufacturing: 75 postings
9. Business Consulting and Services: 71 postings
10. Law Practice: 71 postings

[AI]:
Based on the data, Hospitals and Health Care is not only the top industry by volume of listings but also provides the most generous medical insurance benefits to its employees.

Final Answer:
 Based on the data, Hospitals and Health Care is not only the top industry by volume of listings but also provides the most generous medical insurance benefits to its employees.

=======================================================
PROJECT QUESTION 2: Why do you think Retail has so many job postings but almost no postings listing medical insurance compared to Hospitals and Health Care or Financial Services?
=======================================================
Running agent...

--- Agent Execution Trace ---

[Human]:
Why do you think Retail has so many job postings but almost no postings listing medical insurance compared to Hospitals and Health Care or Financial Services?

[AI]:

Tool Calls: [
  {
    "name": "get_industry_posting_count",
    "args": {
      "industry_name": "Retail"
    },
    "id": "fdde7f90-0722-40e3-afb2-50668fcb2075",
    "type": "tool_call"
  },
  {
    "name": "get_medical_insurance_benefits",
    "args": {},
    "id": "367d4884-2329-49b1-aac7-0a946fad96fe",
    "type": "tool_call"
  },
  {
    "name": "get_industry_posting_count",
    "args": {
      "industry_name": "Hospitals and Health Care"
    },
    "id": "c4c0ca97-ab36-48fa-9ba1-ddb428a6b613",
    "type": "tool_call"
  },
  {
    "name": "get_industry_posting_count",
    "args": {
      "industry_name": "Financial Services"
    },
    "id": "50539036-a4f9-43b9-b892-d824c4b20b4d",
    "type": "tool_call"
  }
]

[Tool]:
Industry 'Retail' has 2383 full-time job postings.

[Tool]:
Top industries by medical insurance benefits postings:
1. Hospitals and Health Care: 235 postings
2. Staffing and Recruiting: 130 postings
3. Financial Services: 121 postings
4. Construction: 116 postings
5. IT Services and IT Consulting: 92 postings
6. Manufacturing: 91 postings
7. Insurance: 77 postings
8. Defense and Space Manufacturing: 75 postings
9. Business Consulting and Services: 71 postings
10. Law Practice: 71 postings

[Tool]:
Industry 'Hospitals and Health Care' has 4557 full-time job postings.

[Tool]:
Industry 'Financial Services' has 2394 full-time job postings.

[AI]:
Based on the data, it appears that Retail has a high number of job postings due to its large size and diverse range of industries within the sector. However, when it comes to medical insurance benefits, Hospitals and Health Care have significantly more postings than other industries.

There are several possible reasons for this:

1. **Industry-specific requirements**: Hospitals and Health Care organizations often require specific medical insurance benefits for their employees, such as coverage for malpractice insurance or specialized medical equipment.
2. **Regulatory compliance**: The healthcare industry is heavily regulated, and employers must comply with various laws and regulations related to employee benefits, including medical insurance.
3. **Employee needs**: Healthcare professionals and support staff often require comprehensive medical insurance coverage due to the nature of their work and the potential risks involved.

In contrast, Retail companies may not have the same level of regulatory requirements or industry-specific needs when it comes to medical insurance benefits. However, this does not mean that Retail companies do not offer medical insurance benefits to their employees; they simply may not be as prominent in job postings.

It's worth noting that these are general trends and may vary depending on specific companies, locations, and industries within each sector.

Final Answer:
 Based on the data, it appears that Retail has a high number of job postings due to its large size and diverse range of industries within the sector. However, when it comes to medical insurance benefits, Hospitals and Health Care have significantly more postings than other industries.

There are several possible reasons for this:

1. **Industry-specific requirements**: Hospitals and Health Care organizations often require specific medical insurance benefits for their employees, such as coverage for malpractice insurance or specialized medical equipment.
2. **Regulatory compliance**: The healthcare industry is heavily regulated, and employers must comply with various laws and regulations related to employee benefits, including medical insurance.
3. **Employee needs**: Healthcare professionals and support staff often require comprehensive medical insurance coverage due to the nature of their work and the potential risks involved.

In contrast, Retail companies may not have the same level of regulatory requirements or industry-specific needs when it comes to medical insurance benefits. However, this does not mean that Retail companies do not offer medical insurance benefits to their employees; they simply may not be as prominent in job postings.

It's worth noting that these are general trends and may vary depending on specific companies, locations, and industries within each sector.

=======================================================
PROJECT QUESTION 3: Given this benefit gap in retail, how can we use this data to advise a job board or recruitment agency on how to target employer outreach or improve listing conversions?
=======================================================
Running agent...

--- Agent Execution Trace ---

[Human]:
Given this benefit gap in retail, how can we use this data to advise a job board or recruitment agency on how to target employer outreach or improve listing conversions?

[AI]:

Tool Calls: [
  {
    "name": "get_industry_posting_count",
    "args": {
      "industry_name": "retail"
    },
    "id": "aeef032e-ba55-4f67-95e8-b781508004db",
    "type": "tool_call"
  }
]

[Tool]:
Industry 'retail' has 2383 full-time job postings.

[AI]:
Based on the data, here are some recommendations for a job board or recruitment agency to target employer outreach and improve listing conversions in the retail industry:

1. **Targeted Job Matching**: Use machine learning algorithms to match job seekers with relevant retail jobs based on their skills, experience, and preferences.
2. **Employer Outreach**: Reach out to top retailers in the industry and offer exclusive job listings, career advice, and recruitment services to attract top talent.
3. **Job Listing Optimization**: Analyze the most effective job listing formats, keywords, and descriptions for retail jobs to improve visibility and appeal to potential candidates.
4. **Employee Referral Programs**: Implement employee referral programs that incentivize current employees to refer their friends and colleagues in the retail industry, reducing recruitment costs and improving time-to-hire.
5. **Industry-Specific Content**: Create content (blog posts, videos, podcasts) that addresses the unique challenges and opportunities in the retail industry, positioning your job board or agency as a thought leader and trusted resource for employers.
6. **Partnerships with Retail Associations**: Collaborate with trade associations and industry groups to access their member directories, job postings, and recruitment resources, expanding your reach and credibility in the retail industry.
7. **Social Media Campaigns**: Leverage social media platforms (e.g., LinkedIn, Twitter) to promote retail jobs, engage with potential candidates, and build your brand as a trusted partner for retail employers.

By implementing these strategies, job boards or recruitment agencies can improve listing conversions and attract top talent in the retail industry.

Final Answer:
 Based on the data, here are some recommendations for a job board or recruitment agency to target employer outreach and improve listing conversions in the retail industry:

1. **Targeted Job Matching**: Use machine learning algorithms to match job seekers with relevant retail jobs based on their skills, experience, and preferences.
2. **Employer Outreach**: Reach out to top retailers in the industry and offer exclusive job listings, career advice, and recruitment services to attract top talent.
3. **Job Listing Optimization**: Analyze the most effective job listing formats, keywords, and descriptions for retail jobs to improve visibility and appeal to potential candidates.
4. **Employee Referral Programs**: Implement employee referral programs that incentivize current employees to refer their friends and colleagues in the retail industry, reducing recruitment costs and improving time-to-hire.
5. **Industry-Specific Content**: Create content (blog posts, videos, podcasts) that addresses the unique challenges and opportunities in the retail industry, positioning your job board or agency as a thought leader and trusted resource for employers.
6. **Partnerships with Retail Associations**: Collaborate with trade associations and industry groups to access their member directories, job postings, and recruitment resources, expanding your reach and credibility in the retail industry.
7. **Social Media Campaigns**: Leverage social media platforms (e.g., LinkedIn, Twitter) to promote retail jobs, engage with potential candidates, and build your brand as a trusted partner for retail employers.

By implementing these strategies, job boards or recruitment agencies can improve listing conversions and attract top talent in the retail industry.


--- Interactive Mode ---
Ask any other questions. Press Ctrl+C or type 'exit' to quit.

